/Users/jemcmull/Dropbox/School/Research/aaaPROJECTS_and_IDEAS/RISK_BP_and_ERM/Code/extract sections from item 1a/extract item1a sections from html file.ipynb
/Users/jemcmull/Dropbox/School/Research/aaaPROJECTS_and_IDEAS/RISK_BP_and_ERM/Code/extract sections from item 1a/extract item1a sections from html file.ipynb

# MAKE A LIST OF HTML FILES I NEED TO CLEAN and SEARCH

In [2]:

import glob


cik_folders = glob.glob('/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/*/*/*/*')


print(len(cik_folders))




2229


In [3]:

for f in cik_folders[0:3]:
    print(f)

/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/789570/000095015309000298/0000950153-09-000298/d10KA.htm
/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/789570/000095015307000441/0000950153-07-000441/d10K.htm
/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/789570/000156459016013639/0001564590-16-013639/d10K.htm


In [99]:
from pathlib import Path
in_dir = Path("filings_raw")
for path in glob.glob('/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/*/*/*/*')[10:11]:
    print(path)

/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/33213/000003321315000004/0000033213-15-000004/d10K.htm


# CLEAN THE HTML FILES TO REMOVE THE XBRL

In [100]:
from bs4 import BeautifulSoup
from pathlib import Path

# Namespaces commonly present in SEC iXBRL filings
XBRL_PREFIXES = {
    "ix", "xbrli", "link", "xlink", "xsi", "dei", "us-gaap", "srt", "num", "country", "currency"
}

# Inline XBRL element local-names that represent visible facts/content.
# We unwrap these so their inner text/HTML remains.
IX_VISIBLE_UNWRAP = {"nonfraction", "nonnumeric", "footnote"}

# Attributes commonly found on iXBRL facts (case-insensitive) that should be removed.
XBRL_ATTRS = {
    "contextref", "unitref", "name", "format", "scale", "sign", "decimals",
    "continuedat", "escape", "footnote", "schemalocation"  # keep lowercase for comparison
}

def strip_xbrl(html: str, keep_html=True) -> str:
    """
    Strip only XBRL/iXBRL markup while preserving regular HTML:
      - Unwrap ix:nonFraction, ix:nonNumeric, ix:footnote (keep content & HTML around them)
      - Remove all other XBRL/inline-XBRL elements entirely
      - Remove XBRL-namespaced attributes and common iXBRL attrs
      - Leave normal HTML (classes, styles, tables, links, etc.) untouched
    """
    soup = BeautifulSoup(html, "lxml")

    # Walk through all elements once
    for tag in soup.find_all(True):
        # 1) Remove XBRL-related attributes but keep normal HTML attributes
        for attr in list(tag.attrs):
            low_attr = attr.lower()
            # Namespaced attr like ix:*, xlink:*, dei:*, etc.
            if ":" in attr:
                prefix = attr.split(":", 1)[0].lower()
                if prefix in XBRL_PREFIXES:
                    del tag[attr]
                    continue
            # Common iXBRL attrs (case-insensitive)
            if low_attr in XBRL_ATTRS:
                del tag[attr]

        # 2) Remove or unwrap XBRL elements
        if ":" in tag.name:
            prefix, local = tag.name.split(":", 1)
            prefix = prefix.lower()
            local = local.lower()

            if prefix in XBRL_PREFIXES:
                if prefix == "ix" and local in IX_VISIBLE_UNWRAP:
                    # Keep the content that was being tagged
                    tag.unwrap()
                else:
                    # Contexts, units, resources, references, hidden blocks, schemas, etc.
                    tag.decompose()

    if keep_html:
        return str(soup)
    else:
        # Plain text (keeps your original HTML out of the way)
        text = soup.get_text(separator="\n")
        lines = [ln.strip() for ln in text.splitlines()]
        return "\n".join([ln for ln in lines if ln])

# Example usage (adjust your input/output dirs as needed)
in_dir = Path("filings_raw")
out_dir = Path("filings_clean")
out_dir.mkdir(parents=True, exist_ok=True)



In [101]:
import glob
from pathlib import Path

for path in glob.glob('/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/*/*/*/*')[21:22]:
    path = Path(path)
    dat_cleaned = strip_xbrl(
        path.read_text(encoding="utf-8", errors="ignore"),
        keep_html=True
    )


In [102]:
print(dat_cleaned[1000:1500])


-family:Times New Roman"><b>SECURITIES EXCHANGE ACT OF 1934 </b></font></p> <p align="center" style="margin-top:0px;margin-bottom:0px"><font size="2" style="font-family:Times New Roman"><b>For the fiscal year ended December 31, 2010 </b></font></p> <p align="center" style="margin-top:0px;margin-bottom:0px"><font size="2" style="font-family:Times New Roman"><b>OR </b></font></p>
<p align="center" style="margin-top:0px;margin-bottom:0px"><font size="2" style="font-family:Times New Roman"><b></b><f


# Remove excessively used HTML tags

In [ ]:
from bs4 import BeautifulSoup
from collections import Counter

def count_html_tag_usage(html_content: str):
    """
    Counts how many times each HTML tag appears in the given HTML content.
    
    Args:
        html_content (str): Raw HTML as a string.
    
    Returns:
        dict: A dictionary mapping tag names to their usage count.
    """
    soup = BeautifulSoup(html_content, "html.parser")
    tags = [tag.name for tag in soup.find_all()]
    return dict(Counter(tags))


# Example usage
if __name__ == "__main__":
    html = """
    <html>
        <head><title>Example</title></head>
        <body>
            <h1>Hello</h1>
            <p>This is a paragraph.</p>
            <div><p>Another paragraph</p></div>
        </body>
    </html>
    """
    
    tag_counts = count_html_tag_usage(dat_cleaned)
    print(tag_counts)

    


{'html': 1, 'head': 1, 'title': 1, 'body': 1, 'h5': 109, 'a': 170, 'p': 2427, 'font': 16900, 'b': 944, 'center': 1, 'table': 233, 'tr': 1886, 'td': 23424, 'u': 7, 'hr': 108, 'br': 215, 'i': 342, 'img': 1, 'sup': 1, 'div': 1}


In [ ]:
from bs4 import BeautifulSoup

def remove_font_tags(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all("font"):
        tag.unwrap()   # removes only the tag but keeps the text
    return str(soup)

dat_cleaned2 = remove_font_tags(dat_cleaned)

In [111]:
print(len(dat_cleaned))
print(len(dat_cleaned2))

2172235
1373838


#  KEEP ONLY PART I 

In [103]:
import re
from bs4 import BeautifulSoup, Tag, NavigableString

# robust matcher: PART II / PART–II / PART — II (case-insensitive, flexible spaces/dashes)
PART_II_RE = re.compile(r'\bpart\s*[-–—]?\s*ii\b', re.I)

def _norm_text(s: str) -> str:
    s = s.replace('\xa0', ' ')
    return re.sub(r'\s+', ' ', s).strip()

def keep_html_until_last_part2(clean_html: str, drop_heading: bool = True) -> str:
    """
    Keep the original HTML structure up to the *last* occurrence of 'PART II'.
    Removes content after that point while preserving containers and markup.
    If drop_heading=True, the 'PART II' element itself is removed; otherwise it's kept.
    """
    soup = BeautifulSoup(clean_html, "lxml")

    # Find the last tag whose *text* contains 'PART II'
    last_tag = None
    for tag in soup.find_all(True):
        try:
            if PART_II_RE.search(_norm_text(tag.get_text(" ", strip=True))):
                last_tag = tag
        except Exception:
            # some malformed tags can throw; just skip them
            continue

    # If no PART II found, return HTML unchanged
    if last_tag is None:
        return str(soup)

    # Helper: remove all following siblings of a node (both tags and strings)
    def _remove_following_siblings(node: Tag):
        for sib in list(node.next_siblings):
            # Extract (not decompose) to avoid touching earlier nodes’ structure
            sib.extract()

    # 1) Remove everything *after* the PART II node inside its parent
    _remove_following_siblings(last_tag)

    # 2) Optionally remove the PART II node itself
    if drop_heading:
        last_tag.extract()

    # 3) Walk up the ancestor chain and prune their following siblings
    p = last_tag.parent if drop_heading else last_tag.parent
    while isinstance(p, Tag):
        _remove_following_siblings(p)
        p = p.parent

    # Done: all HTML *before* PART II is intact; containers remain.
    return str(soup)


In [ ]:

# 2) Truncate *before* last PART II to keep only Part I
part1_html = truncate_before_last_part2(dat_cleaned2, drop_heading=True)

# 3) Now pull text, if needed
from bs4 import BeautifulSoup
text_for_nlp = BeautifulSoup(part1_html, "lxml").get_text(separator="\n")


In [106]:
print(len(path.read_text(encoding="utf-8", errors="ignore")))
print(len(dat_cleaned))
print(len(part1_html))
print(len(text_for_nlp))
print("** "*20)
print(len(path.read_text(encoding="utf-8", errors="ignore")))
print(len(dat_cleaned)/len(path.read_text(encoding="utf-8", errors="ignore")))
print(len(part1_html)/len(path.read_text(encoding="utf-8", errors="ignore")))
print(len(text_for_nlp)/len(path.read_text(encoding="utf-8", errors="ignore")))

2270425
2172235
128439
67358
** ** ** ** ** ** ** ** ** ** ** ** ** ** ** ** ** ** ** ** 
2270425
0.9567525903740489
0.056570465881938405
0.029667573251703976


In [66]:
part1_html[0:500]

'<html><head>\n<title>Form 10-K</title>\n</head>\n<body bgcolor="WHITE">\n<h5 align="left"><a href="#toc">Table of Contents</a></h5>\n<p style="line-height:1px;margin-top:0px;margin-bottom:2px;border-bottom:1pt solid #000000">\xa0</p> <p align="center" style="margin-top:3px;margin-bottom:0px"><font size="4" style="font-family:Times New Roman"><b>SECURITIES AND EXCHANGE COMMISSION </b></font></p> <p align="center" style="margin-top:0px;margin-bottom:0px"><font size="2" style="font-family:Times New Roman">'

## check if risk factor is still in there

In [77]:
import re

# Allow whitespace, &nbsp; and arbitrary HTML tags between tokens
BETWEEN = r'(?:\s|&nbsp;|&#160;|<[^>]*>)*'

# Robust HTML regex: matches "Item 1A. Risk Factors" even with tags/punctuation in between
ITEM_1A_HTML_RE = re.compile(
    r'(?is)\bitem' + BETWEEN +
    r'1' + BETWEEN + r'a\b' +
    r'(?:' + BETWEEN + r'[\.\-–—:])?' +   # optional . — : after "1A"
    BETWEEN + r'risk' + BETWEEN + r'factors\b'
)

def find_all_item1a_html_snippets(html: str, before: int = 20, after: int = 3000):
    """
    Search the RAW HTML (string) for 'Item 1A. Risk Factors' allowing tags/spaces between words,
    and return snippets containing `before` chars before and `after` chars after each match.
    Snippets are raw HTML (will include tags and may start/end mid-tag).
    """
    # Quick sanity check: if no angle brackets, you're passing plain text by mistake
    if "<" not in html or ">" not in html:
        raise ValueError("Input looks like plain text. Pass the RAW HTML string, not soup.get_text().")

    snippets = []
    for m in ITEM_1A_HTML_RE.finditer(html):
        start = max(0, m.start() - before)
        end   = min(len(html), m.end() + after)
        snippets.append(html[start:end])
    return snippets

# --- usage ---
# IMPORTANT: part1_html must be the HTML string (e.g., from keep_html_until_last_part2 / strip_xbrl), not get_text()
snippets = find_all_item1a_html_snippets(part1_html, before=20, after=300000)
for i, snip in enumerate(snippets, 1):
    print(f"\n--- HTML Match {i} ---\n{snip}\n")



--- HTML Match 1 ---
it;font-size:10pt;">Item 1A</font></div></td><td style="vertical-align:top;padding-left:2px;padding-top:2px;padding-bottom:2px;padding-right:2px;"><div style="text-align:left;font-size:10pt;"><font style="font-family:inherit;font-size:10pt;">Risk Factors</font></div></td><td style="vertical-align:bottom;padding-left:2px;padding-top:2px;padding-bottom:2px;padding-right:2px;"><div style="text-align:right;font-size:10pt;"><a href="#sF7FF307C3C81B61A3D2B4546D9D205A0" style="font-family:inherit;font-size:10pt;">19</a></div></td></tr><tr><td style="vertical-align:top;padding-left:2px;padding-top:2px;padding-bottom:2px;padding-right:2px;"><div style="text-align:left;font-size:10pt;"><font style="font-family:inherit;font-size:10pt;">Item 1B</font></div></td><td style="vertical-align:top;padding-left:2px;padding-top:2px;padding-bottom:2px;padding-right:2px;"><div style="text-align:left;font-size:10pt;"><font style="font-family:inherit;font-size:10pt;">Unresolved Staff Comm

In [60]:
print(len(snip))

49731


# PUSH THE CLEANED TEXT TO OPEN AI TO EXTRACT THE RF STATEMENT and DESCRIPTION

In [57]:
prompt_text_path = '/Users/jemcmull/Dropbox/School/Research/aaaPROJECTS_and_IDEAS/RISK_BP_and_ERM/Code/RF_pull_prompt.txt'

with open(prompt_text_path, "r", encoding="utf-8") as f:
    prompt_text = f.read()


In [59]:
print(prompt_text)

You are extracting risk factors from a 10-K. In Item 1A or any section whose heading contains the words "Risk Factors" or "Factors that May Affect," identify every individual risk factor.

STRICT VERBATIM MODE (very important):
- Do not paraphrase, summarize, reword, or correct grammar.
- Preserve capitalization, spelling, numerals, punctuation, dashes, and quotation marks exactly as they appear after HTML entity decoding (e.g., &mdash; -> —, &quot; -> ").
- Remove HTML tags but keep their inner text. Treat <b> and <i> as styling only; do not add or remove words because of styling.
- Convert HTML line breaks (<br>) to single newlines and paragraph tags (<p>, </p>) to paragraph boundaries.
- Inside a single paragraph, collapse any run of whitespace (spaces, tabs, newlines) to a single space.
- Separate paragraphs with two newline characters "\n\n".
- Do not include bullet glyphs or table scaffolding when they are just structural (e.g., "•" cells); capture only the risk text.

DETECTING 

# --> 🔥 Fire up the open AI api

- check account balance here: https://platform.openai.com/settings/organization/billing/overview

In [64]:
import os
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

True

In [68]:
#check that api is up and running
stored_content = ""
client = OpenAI()

test_prompt = 'describe what a robot looks like in 2 sentences.'

stream = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": test_prompt}],
    stream=True,
)
for chunk in stream:
    #print(chunk.choices[0].delta.content or "", end="")
    stored_content += chunk.choices[0].delta.content or ""
print("******* "*10)
print(stored_content)

******* ******* ******* ******* ******* ******* ******* ******* ******* ******* 
A robot looks like a compact machine of metal and polymer, with articulated joints, exposed servos, and cables running between limbs that suggest both strength and precision. Its “face” may be a glowing visor or screen dotted with cameras and sensors, while panels, vents, and indicator LEDs give it an engineered, purposeful appearance.


# Run a test one through open AI

In [93]:
len(part1_html)

3412255

In [90]:
#check that api is up and running
stored_content = ""
client = OpenAI()

#prompt = 'describe what a robot looks like in 2 sentences.'
prompt = prompt_text + part1_html


stream = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}],
    stream=True,
)
for chunk in stream:
    #print(chunk.choices[0].delta.content or "", end="")
    stored_content += chunk.choices[0].delta.content or ""
print("******* "*10)
print(stored_content)

BadRequestError: Error code: 400 - {'error': {'message': 'Input tokens exceed the configured limit of 272000 tokens. Your messages resulted in 914902 tokens. Please reduce the length of the messages.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [72]:
#Turn the call to OpenAI into a function
def ask_openai(part1_html: str):
    #check that api is up and running
    stored_content = ""
    client = OpenAI()

    #prompt = 'describe what a robot looks like in 2 sentences.'
    prompt = prompt_text + part1_html


    stream = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    for chunk in stream:
        #print(chunk.choices[0].delta.content or "", end="")
        stored_content += chunk.choices[0].delta.content or ""
    print("******* "*10)
    print(stored_content)
    return(stored_content)

In [89]:
len(prompt)

130868

# Write function to create cleaned output folder

In [79]:
from pathlib import Path

CLEANED_ROOT = Path("/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM_CLEANED")
ORIGINAL_ROOT = Path("/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM")

def get_cleaned_folder(original_path: str | Path) -> Path:
    original_path = Path(original_path)

    # Compute the relative path under the original root
    rel = original_path.relative_to(ORIGINAL_ROOT)

    # Drop the filename (we only want folders)
    rel_folder = rel.parent

    # Construct the cleaned folder path
    cleaned_folder = CLEANED_ROOT / rel_folder

    # Make sure the directory exists
    cleaned_folder.mkdir(parents=True, exist_ok=True)

    return cleaned_folder

In [80]:
p = path
cleaned_folder = get_cleaned_folder(p)

In [84]:
filename = Path(p).name


'd10K'

# write a loop to pass over all files

Loop will save the cleaned HTML, cleaned text, and the json file to the same folder.

In [115]:
for path_str in glob.glob('/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/*/*/*/*')[10:11]:
    path = Path(path_str)
    print(path)
    cleaned_folder = get_cleaned_folder(path)
    print("loading and cleaning")
    print(len(path.read_text(encoding="utf-8", errors="ignore")))

    out_path = cleaned_folder / f"{filename.split('.')[0]}_OG.htm"
    out_path.write_text(path.read_text(encoding="utf-8", errors="ignore"), encoding="utf-8")

    dat_cleaned = strip_xbrl(
        path.read_text(encoding="utf-8", errors="ignore"),
        keep_html=True
    )
    print(len(dat_cleaned))

    out_path = cleaned_folder / f"{filename.split('.')[0]}_removed_XBRL.htm"
    out_path.write_text(dat_cleaned, encoding="utf-8")
    dat_cleaned = remove_font_tags(dat_cleaned)
    print(len(dat_cleaned))
    out_path = cleaned_folder / f"{filename.split('.')[0]}_removed_fontL.htm"
    out_path.write_text(dat_cleaned, encoding="utf-8")

    print("truncating")
    part1_html = keep_html_until_last_part2(dat_cleaned, drop_heading=True)
    print(len(part1_html))
    out_path = cleaned_folder / f"{filename.split('.')[0]}_truncated_last_PartII.htm"
    out_path.write_text(part1_html, encoding="utf-8")

    print("cleaning")
    text_for_nlp = BeautifulSoup(part1_html, "lxml").get_text(separator="\n")
    print(len(text_for_nlp))
    out_path = cleaned_folder / f"{filename.split('.')[0]}_text_only.txt"

    out_path.write_text(text_for_nlp, encoding="utf-8")

    print("Psst OpenAI...")
    rf_json = ask_openai(part1_html)
    out_path = cleaned_folder / f"{filename.split('.')[0]}_RFs.json"
    out_path.write_text(rf_json, encoding="utf-8")



/Users/jemcmull/Downloads/EDGAR_10Ks_FOR_RISKBP_ERM/33213/000003321315000004/0000033213-15-000004/d10K.htm
loading and cleaning
3831153
3777118
2760611
truncating
2484085
cleaning
431212
Psst OpenAI...


BadRequestError: Error code: 400 - {'error': {'message': 'Input tokens exceed the configured limit of 272000 tokens. Your messages resulted in 669928 tokens. Please reduce the length of the messages.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [75]:
len(text_for_nlp)

462576

In [ ]:
text_for_nlp